02 样本内网格搜索（只看 IS：2012-08 ~ 2019-12）
止损 {6,8,10,12%,无} × 止盈 {15,25,40%,无} × {追踪,固定}。**此处不看样本外。**

在 PyCharm 中选择 Python 内核，点击 **Run All / 运行全部**。最后一格调用 `main()`，无需手动逐个调用函数。


In [1]:
from pathlib import Path
import os
import json

def find_pipeline_dir():
    """支持从项目根目录、pipeline 目录及项目内子目录启动。"""
    roots = ([Path(os.environ['ETF_PROJECT_ROOT']).expanduser()] if os.environ.get('ETF_PROJECT_ROOT') else [])
    roots += [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for candidate in (root / 'pipeline', root):
            if (candidate / '00_engine.ipynb').is_file() and (candidate / '07_run_pipeline.ipynb').is_file():
                return candidate.resolve()
    raise FileNotFoundError('找不到 pipeline 文件夹。请将 PyCharm 工作目录设为项目根目录，或设置 ETF_PROJECT_ROOT。')

PIPELINE_DIR = find_pipeline_dir()

def load_notebook_definitions(filename):
    """独立命名空间加载函数；不执行被加载 notebook 的入口单元格。"""
    path = PIPELINE_DIR / filename
    notebook = json.loads(path.read_text(encoding='utf-8'))
    namespace = {'__name__': 'pipeline_module'}
    for number, item in enumerate(notebook['cells']):
        if item['cell_type'] == 'code' and 'pipeline-entry' not in item.get('metadata', {}).get('tags', []):
            exec(compile(''.join(item['source']), f'{path}:cell{number}', 'exec'), namespace)
    return namespace

_engine = load_notebook_definitions('00_engine.ipynb')
for _name, _value in _engine.items():
    if not _name.startswith('_'):
        globals()[_name] = _value


In [2]:
def evaluate_grid(runs):
    rows=[{'trail':c[0],'sl':c[1],'tp':c[2],**metrics(runs[c][0],a=IS_START,b=IS_END)} for c in combos]
    g=pd.DataFrame(rows)
    for trailing in [True,False]:
        sub=g[g.trail==trailing]
        print('== 追踪止损' if trailing else '== 固定止损','（行=止损，列=止盈）')
        for col in ['Sharpe','MDD']:
            print('IS',col); display(sub.pivot(index='sl',columns='tp',values=col).round(2))
    return {'grid': g}


In [3]:
def main(csv_path=None, context=None, runs=None):
    """本 notebook 的一键入口；返回表格、净值等结果。"""
    context = prepare_context(csv_path) if context is None else context
    O, H, L, C, tg = context
    runs = run_all(*context) if runs is None else runs
    return evaluate_grid(runs)


In [4]:
# 修改数据路径后，运行全部单元格。None 表示自动查找项目 data/etf_daily.csv。
CSV_PATH = None
results = main(csv_path=CSV_PATH)


数据区间: 2011-10-03 ~ 2026-10-07 | 信号分布: {'TQQQ': 2904, 'TECL': 249, 'TECS': 203, 'BSV': 150, 'UVXY': 133, 'SPXL': 17}


== 追踪止损 （行=止损，列=止盈）
IS Sharpe


tp,NaN,0.15,0.25,0.40
sl,,,,
NaN,0.99,0.99,1.08,1.04
0.06,0.71,0.72,0.78,0.80
0.08,0.56,0.62,0.64,0.65
0.10,0.55,0.63,0.64,0.64
0.12,0.65,0.78,0.75,0.73


IS MDD


tp,NaN,0.15,0.25,0.40
sl,,,,
NaN,-0.74,-0.64,-0.63,-0.74
0.06,-0.49,-0.42,-0.46,-0.49
0.08,-0.56,-0.50,-0.54,-0.56
0.10,-0.59,-0.53,-0.58,-0.59
0.12,-0.64,-0.57,-0.62,-0.64


== 固定止损 （行=止损，列=止盈）
IS Sharpe


tp,NaN,0.15,0.25,0.40
sl,,,,
0.06,0.76,0.82,0.94,0.83
0.08,0.62,0.66,0.78,0.69
0.10,0.76,0.85,0.93,0.82
0.12,0.80,0.85,0.94,0.86


IS MDD


tp,NaN,0.15,0.25,0.40
sl,,,,
0.06,-0.69,-0.53,-0.52,-0.69
0.08,-0.75,-0.65,-0.61,-0.75
0.10,-0.73,-0.58,-0.57,-0.73
0.12,-0.73,-0.62,-0.61,-0.73
